In [2]:
!pip install -q language-tool-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 1.5 MB/s eta 0:00:00a 0:00:01


In [3]:
import torch
import librosa
import numpy as np
import pandas as pd
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import re
import spacy
import language_tool_python
from collections import Counter
import os

In [4]:
nlp = spacy.load("en_core_web_sm")

grammar_tool = language_tool_python.LanguageTool("en-US")

In [5]:
file_dir = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/"

In [6]:
processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")   

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.06GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

In [7]:
def transcribe(file_path):
    audio, _ = librosa.load(file_path, sr=16000)

    input_features = processor(audio, sampling_rate=16000, return_tensors="pt").input_features

    with torch.no_grad():
        output_ids = model.generate(input_features)

    return processor.batch_decode(output_ids, skip_special_tokens=True)[0]

In [8]:
def extract_features(file_path, transcription=None):

    audio, sr = librosa.load(file_path, sr=16000)

    features = {}

    # --------------------------------------------------
    # Speech / Silence Detection
    # --------------------------------------------------

    speech_segments = librosa.effects.split(
        audio,
        top_db=30
    )

    total_duration = len(audio) / sr

    speech_duration = sum(
        (end - start) / sr
        for start, end in speech_segments
    )

    features["speech_duration"] = float(speech_duration)

    # Proportion of audio containing speech
    features["speech_ratio"] = (
        float(speech_duration / total_duration)
        if total_duration > 0
        else 0.0
    )

    # --------------------------------------------------
    # Pauses
    # --------------------------------------------------

    pauses = []

    for i in range(len(speech_segments) - 1):

        gap = (
            speech_segments[i + 1][0]
            - speech_segments[i][1]
        ) / sr

        # Ignore extremely small gaps
        if gap > 0.1:
            pauses.append(gap)

    features["num_pauses"] = int(len(pauses))

    features["pause_mean"] = (
        float(np.mean(pauses))
        if pauses
        else 0.0
    )

    features["pause_max"] = (
        float(np.max(pauses))
        if pauses
        else 0.0
    )

    # --------------------------------------------------
    # Total Silence
    # --------------------------------------------------

    features["silence_duration"] = float(
        total_duration - speech_duration
    )

    # --------------------------------------------------
    # Speaking Rate
    # --------------------------------------------------

    if transcription and speech_duration > 0:

        words = transcription.split()

        features["speaking_rate"] = float(
            len(words) / speech_duration
        )

    else:

        features["speaking_rate"] = 0.0

    return features

In [9]:
def extract_text_features(transcription):

    features = {}

    # =========================================================
    # CLEAN TEXT
    # =========================================================

    text = str(transcription).strip()
    text = re.sub(r"\s+", " ", text)

    doc = nlp(text)

    words = [
        token for token in doc
        if token.is_alpha
    ]

    n_words = len(words)

    sentences = list(doc.sents)
    n_sentences = len(sentences)

    # =========================================================
    # SENTENCE STRUCTURE
    # =========================================================

    features["word_count"] = n_words
    features["sentence_count"] = n_sentences

    sentence_lengths = [
        sum(1 for token in sent if token.is_alpha)
        for sent in sentences
    ]

    if sentence_lengths:

        features["avg_sentence_length"] = float(
            np.mean(sentence_lengths)
        )

        features["std_sentence_length"] = float(
            np.std(sentence_lengths)
        )

    else:

        features["avg_sentence_length"] = 0.0
        features["std_sentence_length"] = 0.0

    # =========================================================
    # LEXICAL DIVERSITY
    # =========================================================

    unique_words = set(
        token.lemma_.lower()
        for token in words
    )

    features["unique_word_ratio"] = (
        len(unique_words) / n_words
        if n_words > 0 else 0.0
    )

    # =========================================================
    # VERB / TENSE FEATURES
    # =========================================================

    verbs = [
        token for token in words
        if token.pos_ in {"VERB", "AUX"}
    ]

    features["verb_ratio"] = (
        len(verbs) / n_words
        if n_words > 0 else 0.0
    )

    tense_counts = Counter()

    for token in verbs:

        tense = token.morph.get("Tense")

        if tense:
            tense_counts[tense[0]] += 1

    total_tensed_verbs = sum(
        tense_counts.values()
    )

    features["past_tense_ratio"] = (
        tense_counts["Past"] / total_tensed_verbs
        if total_tensed_verbs > 0 else 0.0
    )

    features["present_tense_ratio"] = (
        tense_counts["Pres"] / total_tensed_verbs
        if total_tensed_verbs > 0 else 0.0
    )

    features["tense_variety"] = len(
        tense_counts
    )

    # =========================================================
    # SYNTAX COMPLEXITY
    # =========================================================

    dependency_depths = [
        len(list(token.ancestors))
        for token in words
    ]

    if dependency_depths:

        features["avg_dependency_depth"] = float(
            np.mean(dependency_depths)
        )

        features["max_dependency_depth"] = float(
            np.max(dependency_depths)
        )

    else:

        features["avg_dependency_depth"] = 0.0
        features["max_dependency_depth"] = 0.0

    # =========================================================
    # CLAUSE COMPLEXITY
    # =========================================================

    subordinate_deps = {
        "advcl",
        "ccomp",
        "xcomp",
        "acl",
        "relcl"
    }

    features["subordinate_clause_count"] = sum(
        token.dep_ in subordinate_deps
        for token in doc
    )

    features["relative_clause_count"] = sum(
        token.dep_ == "relcl"
        for token in doc
    )

    features["coordination_count"] = sum(
        token.dep_ == "conj"
        for token in doc
    )

    # =========================================================
    # INCOMPLETE / FRAGMENTED SENTENCES
    # =========================================================

    fragment_count = 0

    for sent in sentences:

        sent_tokens = [
            token for token in sent
            if token.is_alpha
        ]

        has_verb = any(
            token.pos_ in {"VERB", "AUX"}
            for token in sent_tokens
        )

        if len(sent_tokens) >= 3 and not has_verb:
            fragment_count += 1

    features["fragment_ratio"] = (
        fragment_count / n_sentences
        if n_sentences > 0 else 0.0
    )

    # =========================================================
    # GRAMMAR ERRORS
    # =========================================================

    try:

        matches = grammar_tool.check(text)

        grammar_errors = []

        for match in matches:

            category = str(
                getattr(match, "category", "")
            ).upper()

            # Ignore spelling/capitalization-only issues
            if (
                "TYPOS" not in category
                and "SPELL" not in category
                and "CASING" not in category
            ):
                grammar_errors.append(match)

        features["grammar_error_count"] = len(
            grammar_errors
        )

        features["grammar_error_rate"] = (
            len(grammar_errors) / n_words
            if n_words > 0 else 0.0
        )

    except Exception:

        features["grammar_error_count"] = 0.0
        features["grammar_error_rate"] = 0.0

    return features

In [10]:
def extract_features(file_path, transcription=None):

    audio, sr = librosa.load(
        file_path,
        sr=16000
    )

    features = {}

    # =========================================================
    # SPEECH / SILENCE SEGMENTS
    # =========================================================

    speech_segments = librosa.effects.split(
        audio,
        top_db=30
    )

    total_duration = len(audio) / sr

    speech_duration = sum(
        (end - start) / sr
        for start, end in speech_segments
    )

    features["speech_duration"] = float(
        speech_duration
    )

    features["speech_ratio"] = (
        float(speech_duration / total_duration)
        if total_duration > 0
        else 0.0
    )

    # =========================================================
    # PAUSES
    # =========================================================

    pauses = []

    for i in range(len(speech_segments) - 1):

        gap = (
            speech_segments[i + 1][0]
            - speech_segments[i][1]
        ) / sr

        # Gaps > 100 ms treated as pauses
        if gap > 0.1:
            pauses.append(gap)

    features["num_pauses"] = int(
        len(pauses)
    )

    features["pause_mean"] = (
        float(np.mean(pauses))
        if pauses
        else 0.0
    )

    features["pause_max"] = (
        float(np.max(pauses))
        if pauses
        else 0.0
    )

    # =========================================================
    # SILENCE
    # =========================================================

    features["silence_duration"] = float(
        total_duration - speech_duration
    )

    # =========================================================
    # SPEAKING RATE
    # =========================================================

    if transcription and speech_duration > 0:

        word_count = len(
            transcription.split()
        )

        features["speaking_rate"] = float(
            word_count / speech_duration
        )

    else:

        features["speaking_rate"] = 0.0

    return features

In [11]:
def process_audio(filepath):
    
    # -----------------------------
    # 1. Transcribe audio
    # -----------------------------
    transcription = transcribe(filepath)

    # -----------------------------
    # 2. Extract audio features
    # -----------------------------
    audio_features = extract_features(
        filepath,
        transcription=transcription
    )

    # -----------------------------
    # 3. Extract NLP / grammar features
    # -----------------------------
    text_features = extract_text_features(
        transcription
    )

    # -----------------------------
    # 4. Combine features
    # -----------------------------
    features = {
        **audio_features,
        **text_features
    }

    return features

In [ ]:
train_dir = f"{file_dir}/train"

features = []

for index,filename in enumerate(os.listdir(train_dir)):

    filepath = os.path.join(train_dir, filename)

    if os.path.isfile(filepath):
        features.append(process_audio(filepath))

    print(f"Done feature ext for {index}")

with open("output.jsonl", "w") as f:
    for index, entry in enumerate(features):
        print(f"Done feature insertion for {index}")
        f.write(json.dumps(entry) + "\n")

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

Done feature ext for 0
Done feature ext for 1
Done feature ext for 2
Done feature ext for 3
Done feature ext for 4
Done feature ext for 5
Done feature ext for 6
Done feature ext for 7
Done feature ext for 8
Done feature ext for 9
Done feature ext for 10
Done feature ext for 11
Done feature ext for 12
Done feature ext for 13
Done feature ext for 14
Done feature ext for 15
Done feature ext for 16
Done feature ext for 17
Done feature ext for 18
Done feature ext for 19
Done feature ext for 20
Done feature ext for 21
Done feature ext for 22
Done feature ext for 23
Done feature ext for 24
Done feature ext for 25
Done feature ext for 26
Done feature ext for 27
Done feature ext for 28
Done feature ext for 29
Done feature ext for 30
Done feature ext for 31
Done feature ext for 32
Done feature ext for 33
Done feature ext for 34
Done feature ext for 35
Done feature ext for 36
Done feature ext for 37
Done feature ext for 38
Done feature ext for 39
Done feature ext for 40
Done feature ext for 41
Do

In [ ]:
print(features)